# raw → `df_ts.parquet`

`data-v2/raw/`만 읽어서 시계열 모델 입력 `data-v2/input/df_ts.parquet`를 만듭니다. 로직은 모두 이 노트북 안에 있고, 다른 모듈이나 중간 캐시는 쓰지 않습니다. 실행할 때마다 SUVI NetCDF에서 다시 추출합니다.

`df.parquet`가 (`set`, `origin`, `h`) 긴 형식의 요약 특징 표라면, `df_ts.parquet`는 **매시 칸 1행**의 시계열 표입니다. 시계열 모델(`research/_module/seq.py`)이 origin O의 lookback 창 O−119…O를 이 표에서 잘라 씁니다.

| 단계 | 내용 |
|---|---|
| 0 | 설정과 시간 칸 규칙 |
| 1 | ACE 월별 txt → 시간 칸 표, 정답 `y` |
| 2 | 시간 축 |
| 3 | SUVI NetCDF → 프레임별 경도 stripe 프로파일 |
| 4 | ENLIL 피처 |
| 5 | 조립·저장 |
| 6 | 검증 |

**규칙** (`raw_to_input_df.ipynb`와 같음)
- 입수지연·결측·이상치 처리는 하지 않습니다. 앞값 채움, 이전 프레임으로 되돌아가기, 마스크, clip·log 변환이 없습니다(모델 쪽 `seq.py`가 합니다).
- **시간 칸**: 시각 t는 칸 `floor(t, 1h) + 1h`에 들어갑니다([H−1h, H) → H). 한 칸에 여러 개면 가장 늦은 것을 씁니다. origin O는 칸 ≤ O만 쓸 수 있습니다.
- **정답 `y`**: 행 `time` = T의 `y`는 T 라벨 SWEPAM 속도(status 0, 유한한 양수)이고 시각을 이동하지 않으며 채우지 않습니다. 같은 행의 `ace_speed_kms`는 칸 T 값(= 라벨 T−1h 행)이라 `y`와 한 시간 어긋납니다.
- 열 이름에 `ch`를 쓰지 않습니다.

## 0. 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import json, re, time, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import ndimage

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40)

ROOT = Path("/home/t-lab01/sunrun")
DATA = ROOT / "data-v2"
ACE_DIR = DATA / "raw/ACE/monthly-txt"
NC_DIR = DATA / "raw/SUVI/nc/g18"
ENLIL_DIR = DATA / "raw/ENLIL/donki"
OUT = Path(os.environ.get("DF_OUT", DATA / "input"))       # df_ts.parquet, MANIFEST_ts.json
SMOKE = os.environ.get("DF_SMOKE") == "1"                   # 짧은 구간만 (확인용)
WORKERS = int(os.environ.get("DF_WORKERS", "12"))

H = 72
MASK_HOURS = 144
HOUR = pd.Timedelta(hours=1)
PERIODS = {  # 이름 -> (첫 query origin, 마지막 query origin, 학습 cutoff)
    "F1": ("2025-12-31T23:00Z", "2026-03-28T23:00Z", "2025-12-29T00:00Z"),
    "F2": ("2026-03-31T23:00Z", "2026-06-27T23:00Z", "2026-03-29T00:00Z"),
    "F3": ("2026-06-30T23:00Z", "2026-09-26T23:00Z", "2026-06-29T00:00Z"),
}
QUERY_SPAN = ("2025-12-31T23:00Z", "2026-09-26T23:00Z")
TRAIN_SPAN = ("2022-10-01T00:00Z", str(pd.Timestamp(PERIODS["F3"][2]) - pd.Timedelta(hours=MASK_HOURS)))
if SMOKE:
    TRAIN_SPAN = ("2026-05-20T00:00Z", "2026-05-21T23:00Z")
    QUERY_SPAN = ("2026-05-28T00:00Z", "2026-05-29T23:00Z")

train_origins = pd.date_range(*TRAIN_SPAN, freq="h")
query_origins = pd.date_range(*QUERY_SPAN, freq="h")
all_origins = train_origins.union(query_origins)


def bin_hour(t):
    # 시각 -> 시간 칸: [H-1h, H) -> H
    return pd.DatetimeIndex(pd.to_datetime(t, utc=True)).floor("h") + HOUR


for raw_t, want in [("2026-05-01T19:49Z", "2026-05-01T20:00Z"), ("2026-05-01T19:59:59Z", "2026-05-01T20:00Z"),
                    ("2026-05-01T19:00Z", "2026-05-01T20:00Z"), ("2026-05-01T20:00Z", "2026-05-01T21:00Z")]:
    assert bin_hour([raw_t])[0] == pd.Timestamp(want), (raw_t, want)

print("SMOKE" if SMOKE else "FULL", "| train origin", len(train_origins), TRAIN_SPAN, "| query origin", len(query_origins), QUERY_SPAN)
print("출력:", OUT, "| workers:", WORKERS)

FULL | train origin 32665 ('2022-10-01T00:00Z', '2026-06-23 00:00:00+00:00') | query origin 6457 ('2025-12-31T23:00Z', '2026-09-26T23:00Z')
출력: /home/t-lab01/sunrun/data-v2/input | workers: 20


## 1. ACE: 월별 txt → 시간 칸 표, 정답 `y`

`raw_to_input_df.ipynb`의 1단계와 같은 코드입니다. NASA 월별 1h 파일 5종(`swepam, mag, epam, sis, loc`)을 읽습니다.
- 피처용 표 `ace_bin`: 라벨 시각 t → 칸 t + 1h. status가 0이 아니거나 결측 표시값이면 NaN입니다.
- `y_series`: 라벨 시각 그대로, SWEPAM `status == 0`이고 유한한 양수인 속도.

In [2]:
# 제품 -> (시각 뒤 필드 순서, 결측 표시값). 필드: (열 이름 | None, status 열 이름 | None)
# 시각 필드 6개: YR MO DA HHMM MJD SEC
ACE_SPEC = {
    "swepam": ([("S", None), ("ace_density_cm3", "S"), ("ace_speed_kms", "S"), ("ace_temperature_k", "S")],
               {"ace_density_cm3": -9999.9, "ace_speed_kms": -9999.9, "ace_temperature_k": -1.0e5}),
    "mag": ([("S", None), ("ace_bx_gsm_nt", "S"), ("ace_by_gsm_nt", "S"), ("ace_bz_gsm_nt", "S"), ("ace_bt_nt", "S"),
             ("ace_lat_gsm_deg", "S"), ("ace_lon_gsm_deg", "S")], -999.9),
    "epam": ([("Se", None), ("ace_epam_e_channel1", "Se"), ("ace_epam_e_channel2", "Se"), ("Sp", None),
              ("ace_epam_p_channel1", "Sp"), ("ace_epam_p_channel2", "Sp"), ("ace_epam_p_channel3", "Sp"),
              ("ace_epam_p_channel4", "Sp"), ("ace_epam_p_channel5", "Sp"), ("ace_epam_anis_index", "Sp")],
             {"ace_epam_anis_index": -1.0, "*": -1.0e5}),
    "sis": ([("S10", None), ("ace_sis_gt10_flux", "S10"), ("S30", None), ("ace_sis_gt30_flux", "S30")], -1.0e5),
    "loc": ([("ace_x_gse_re", None), ("ace_y_gse_re", None), ("ace_z_gse_re", None)], -999.9),
}
ACE_LABEL = {  # 원문 헤더의 이름 (문서용)
    "ace_density_cm3": "SWEPAM proton density", "ace_speed_kms": "SWEPAM bulk speed", "ace_temperature_k": "SWEPAM ion temperature",
    "ace_bx_gsm_nt": "MAG Bx GSM", "ace_by_gsm_nt": "MAG By GSM", "ace_bz_gsm_nt": "MAG Bz GSM", "ace_bt_nt": "MAG Bt",
    "ace_lat_gsm_deg": "MAG lat GSM", "ace_lon_gsm_deg": "MAG long GSM",
    "ace_epam_e_channel1": "EPAM electron 38-53 keV", "ace_epam_e_channel2": "EPAM electron 175-315 keV",
    "ace_epam_p_channel1": "EPAM proton 47-68 keV", "ace_epam_p_channel2": "EPAM proton 115-195 keV",
    "ace_epam_p_channel3": "EPAM proton 310-580 keV", "ace_epam_p_channel4": "EPAM proton 795-1193 keV",
    "ace_epam_p_channel5": "EPAM proton 1060-1900 keV", "ace_epam_anis_index": "EPAM anisotropy index",
    "ace_sis_gt10_flux": "SIS proton > 10 MeV", "ace_sis_gt30_flux": "SIS proton > 30 MeV",
    "ace_x_gse_re": "LOC X GSE", "ace_y_gse_re": "LOC Y GSE", "ace_z_gse_re": "LOC Z GSE",
}


def read_ace(product):
    fields, missing = ACE_SPEC[product]
    names = [n for n, _ in fields]
    rows, files = [], sorted(ACE_DIR.glob(f"*_ace_{product}_1h.txt"))
    for order, path in enumerate(files):
        for line in path.read_text().splitlines():
            if not line or line[0] in ":#":
                continue
            parts = line.split()
            if len(parts) != 6 + len(fields):
                raise ValueError(f"{path.name}: 필드 수 {len(parts)}: {line!r}")
            rows.append([order] + parts[:4] + parts[6:])
    raw = pd.DataFrame(rows, columns=["file", "yr", "mo", "da", "hhmm"] + names)
    hhmm = raw["hhmm"].astype(int)
    t = pd.to_datetime(dict(year=raw.yr.astype(int), month=raw.mo.astype(int), day=raw.da.astype(int)), utc=True)
    raw.insert(0, "label_utc", t + pd.to_timedelta(hhmm // 100, unit="h") + pd.to_timedelta(hhmm % 100, unit="m"))
    vals = raw[names].astype(float)
    vals.insert(0, "label_utc", raw["label_utc"])
    vals.insert(1, "file", raw["file"])
    conflict = int(vals.drop(columns="file").drop_duplicates().duplicated("label_utc").sum())
    vals = vals.sort_values(["label_utc", "file"]).drop_duplicates("label_utc", keep="last").set_index("label_utc")
    clean = pd.DataFrame(index=vals.index)
    for name, status in fields:
        if not name.startswith("ace_"):
            continue
        v = vals[name].copy()
        miss = missing.get(name, missing.get("*")) if isinstance(missing, dict) else missing
        bad = np.isclose(v, miss, rtol=0, atol=1e-6) | ~np.isfinite(v)
        if status is not None:
            bad |= vals[status].ne(0)
        clean[name] = v.where(~bad)
    return clean, vals, {"files": len(files), "rows": len(rows), "labels": len(vals), "conflicting_duplicates": conflict}


ace_parts, ace_raw, ace_info = {}, {}, {}
for product in ACE_SPEC:
    ace_parts[product], ace_raw[product], ace_info[product] = read_ace(product)
ace_label = pd.concat(ace_parts.values(), axis=1).sort_index()      # 라벨 시각 기준, status/결측 표시 -> NaN
assert ace_label.index.is_unique

# 피처용: 라벨 시각 -> 칸 (라벨이 모두 정각이라 칸 = 라벨 + 1h, 한 칸에 한 행)
ace_bin = ace_label.copy()
ace_bin.index = bin_hour(ace_bin.index)
ace_bin = ace_bin[~ace_bin.index.duplicated(keep="last")]
ace_bin = ace_bin.reindex(pd.date_range(ace_bin.index.min(), ace_bin.index.max(), freq="h"))

# y용: 라벨 시각 그대로, SWEPAM status == 0 이고 유한한 양수
sw = ace_raw["swepam"]
y_series = sw["ace_speed_kms"].where(sw["S"].eq(0) & np.isfinite(sw["ace_speed_kms"]) & (sw["ace_speed_kms"] > 0))
y_series = y_series.reindex(pd.date_range(y_series.index.min(), y_series.index.max(), freq="h"))

display(pd.DataFrame(ace_info).T)
print("ACE 라벨 시각:", ace_label.index.min(), "~", ace_label.index.max(), "| 칸:", ace_bin.index.min(), "~", ace_bin.index.max())
print("y 유효:", int(y_series.notna().sum()), "/", len(y_series), f"({y_series.notna().mean():.1%})")
display(pd.DataFrame({"원문": ACE_LABEL, "NaN 비율": ace_label.isna().mean().round(3)}))

,files,rows,labels,conflicting_duplicates
swepam,52,37417,37417,0
mag,52,37417,37417,0
epam,52,37417,37417,0
sis,52,37417,37417,0
loc,52,37440,37440,0


ACE 라벨 시각: 2022-08-01 00:00:00+00:00 ~ 2026-11-07 23:00:00+00:00 | 칸: 2022-08-01 01:00:00+00:00 ~ 2026-11-08 00:00:00+00:00
y 유효: 25248 / 37417 (67.5%)


,원문,NaN 비율
ace_density_cm3,SWEPAM proton density,0.323
ace_speed_kms,SWEPAM bulk speed,0.326
ace_temperature_k,SWEPAM ion temperature,0.326
ace_bx_gsm_nt,MAG Bx GSM,0.174
ace_by_gsm_nt,MAG By GSM,0.174
ace_bz_gsm_nt,MAG Bz GSM,0.174
ace_bt_nt,MAG Bt,0.174
ace_lat_gsm_deg,MAG lat GSM,0.174
ace_lon_gsm_deg,MAG long GSM,0.174
ace_epam_e_channel1,EPAM electron 38-53 keV,0.145


## 2. 시간 축

표의 행은 매시 칸 `time`입니다. 전체 실행은 첫 ACE 칸부터 마지막 query origin + 72h까지입니다. lookback 120h와 27일 재귀(`time + 72h − 28일`)가 첫 학습 origin(2022-10-01)에서도 표 안에 들어옵니다.

In [3]:
t_first = ace_bin.index.min() if not SMOKE else all_origins[0] - pd.Timedelta(days=6)
t_last = all_origins[-1] + pd.Timedelta(hours=H)
times = pd.date_range(t_first, t_last, freq="h")
assert times[0] <= train_origins[0] - pd.Timedelta(days=28) or SMOKE
ace_ts = ace_bin.reindex(times)
y_ts = y_series.reindex(times)
ACE_VARS = list(ace_ts.columns)
print(f"시간 축 {len(times):,}칸: {times[0]} ~ {times[-1]} | ACE 변수 {len(ACE_VARS)} | y 유효 {y_ts.notna().mean():.1%}")

시간 축 36,503칸: 2022-08-01 01:00:00+00:00 ~ 2026-09-29 23:00:00+00:00 | ACE 변수 22 | y 유효 67.4%


## 3. SUVI: NetCDF → 프레임별 경도 stripe 프로파일

`raw/SUVI/nc/g18/`의 GOES-18 Fe195 L1b 파일을 씁니다. 파일 이름의 `_e…`(관측 끝 `obs_end`)로 칸 `bin_hour(obs_end)`를 정하고, 한 칸에 여러 개면 `obs_end`가 가장 늦은 것을 씁니다. **칸 T의 행에는 칸 T 프레임만** 들어가고, 프레임이 없으면 SUVI 열이 모두 NaN입니다.

프레임 한 장에서 폭 10° stripe를 경도 방향으로 옮겨 가며 통계를 뽑습니다(유효 픽셀 = 디스크 0.98 R☉ 안, `DQF == 0`, 유한한 비음수 radiance).
- stripe `k`(= −72 … 143)의 중심 경도는 λ_k = 0.55°/h × (k − `img_age_h`)입니다. 서쪽이 +이고, 태양 자전 0.55°/h 기준으로 **칸 시각보다 k시간 전에 중앙 자오선을 지난 경도**입니다(k < 0이면 앞으로 지날 경도).
- stripe 영역 = |경도 − λ_k| ≤ 5°, |위도| ≤ 30°. 기존 `strip_v*`가 15° 셀의 겹침 가중 평균이었던 것과 달리 픽셀에서 직접 계산합니다.
- 통계 3종(기존 strip과 같은 뜻)
  - `suvi_strip_logmed_k*`: 유효 픽셀 log1p radiance의 중앙값
  - `suvi_strip_dark_k*`: 디스크 중앙값의 0.5배보다 어두운 유효 픽셀 비율
  - `suvi_strip_cover_k*`: 영역 픽셀 중 유효 픽셀 비율(영역이 비면 0)
- 열 이름의 k는 `m072`(−72) … `p000` … `p143`입니다.
- `img_age_h` = 칸 − `obs_end`(시간, 0 초과 1 이하), `suvi_disk_log_median` = 디스크 유효 픽셀 log1p 중앙값.

같은 프레임에서 **프레임 요약 327열**도 함께 뽑습니다. `raw_to_input_df.ipynb`와 같은 코드·같은 이름입니다.
- 셀 통계 147열 `suvi_disk_*`, `suvi_lonII_latJ_*`: 디스크와 경도 12칸 × 위도 3칸의 `log_median`·`log_q10`·`dark_fraction`·`valid_fraction`.
- 어두운 영역 면적 180열 `suvi_t035_*`, `suvi_t045_*`, `suvi_t055_*`: 임계 0.35/0.45/0.55의 면적·개수·상위 3개 영역·자오선/적도 면적·36칸 면적.
- h마다 다른 `strip_v*`(속도별 strip)는 `lat1` 셀 통계와 `img_age_h`의 함수라 표에 넣지 않고, 쓰는 쪽(`research/_module/seq.py`)이 계산합니다.

좌표 계산식은 `raw_to_input_df.ipynb`의 `cell_stats`와 같습니다.

In [4]:
SUVI_NAME = re.compile(r"_s(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_e(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_c")


def stamp(g):
    year, doy, hh, mm, ss, tenth = (int(x) for x in g)
    return pd.Timestamp(year=year, month=1, day=1, tz="UTC") + pd.Timedelta(days=doy - 1, hours=hh, minutes=mm,
                                                                             seconds=ss, milliseconds=100 * tenth)


def read_suvi_frame(path):
    # RAD를 물리 radiance로 한 번만 디코딩 (h5py, 자동 scale 없음). fill 값은 NaN
    with h5py.File(path, "r") as nc:
        var = nc["RAD"]
        raw = np.asarray(var[...])
        rad = raw.astype(np.float64)
        invalid = ~np.isfinite(rad)
        for name in ("_FillValue", "missing_value"):
            if name in var.attrs:
                invalid |= raw == np.asarray(var.attrs[name]).reshape(-1)[0]
        rad = rad * float(np.asarray(var.attrs.get("scale_factor", 1)).item())
        rad += float(np.asarray(var.attrs.get("add_offset", 0)).item())
        rad[invalid] = np.nan
        header = dict(nc.attrs)
        for key in ("CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "CDELT1", "CDELT2", "PC1_1", "PC1_2", "PC2_1", "PC2_2"):
            if key in nc:
                header[key] = np.asarray(nc[key][...]).item()
        return rad.squeeze(), np.asarray(nc["DQF"][...]).squeeze(), header


def cell_stats(rad, dqf, header):
    # 디스크와 경도 12 x 위도 3 셀의 log1p radiance 통계 (148열)
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    h = {str(k).upper(): v for k, v in header.items()}

    def number(key, default=None):
        value = h.get(key, default)
        if value is None:
            raise ValueError(f"missing SUVI geometry: {key}")
        return float(np.asarray(value).item())

    yy, xx = np.indices(rad.shape, dtype=float)
    xx -= number("CRPIX1") - 1
    yy -= number("CRPIX2") - 1
    theta = np.deg2rad(number("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[number("PC1_1"), number("PC1_2")], [number("PC2_1"), number("PC2_2")]])
    rsun = number("RSUN")
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    x = (matrix[0, 0] * xx + matrix[0, 1] * yy) / rsun
    y = (matrix[1, 0] * xx + matrix[1, 1] * yy) / rsun
    disk = x * x + y * y <= 0.98 ** 2
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    z = np.sqrt(np.maximum(0, 1 - x * x - y * y))
    b0 = np.deg2rad(number("SOLAR_B0", 0))
    lat = np.arcsin(np.clip(y * np.cos(b0) + z * np.sin(b0), -1, 1))
    lon = np.arctan2(x, z * np.cos(b0) - y * np.sin(b0))
    longitude = np.clip(np.floor((lon + np.pi / 2) / np.pi * 12), 0, 11).astype(int)
    latitude = np.clip(np.floor((lat + np.pi / 2) / np.pi * 3), 0, 2).astype(int)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    result = {}
    for name, region in [("disk", disk)] + [(f"lon{i:02d}_lat{j}", disk & (longitude == i) & (latitude == j))
                                            for i in range(12) for j in range(3)]:
        values = rad[valid & region]
        log = np.log1p(values)
        result[f"suvi_{name}_log_median"] = float(np.median(log)) if len(log) else np.nan
        result[f"suvi_{name}_log_q10"] = float(np.quantile(log, .1)) if len(log) else np.nan
        result[f"suvi_{name}_dark_fraction"] = float(np.mean(values < .5 * reference)) if len(log) else np.nan
        result[f"suvi_{name}_valid_fraction"] = float(len(values) / region.sum()) if region.any() else 0.
    return result


THRESHOLDS = (0.35, 0.45, 0.55)


def dark_geometry(shape, h):
    rsun = h["RSUN"]
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    y, x = np.indices(shape, dtype=float)
    x -= h["CRPIX1"] - 1
    y -= h["CRPIX2"] - 1
    theta = np.deg2rad(h.get("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]])
    xx = (matrix[0, 0] * x + matrix[0, 1] * y) / rsun
    yy = (matrix[1, 0] * x + matrix[1, 1] * y) / rsun
    disk = xx ** 2 + yy ** 2 <= .98 ** 2
    z = np.sqrt(np.maximum(1e-8, 1 - xx ** 2 - yy ** 2))
    b0 = np.deg2rad(h.get("SOLAR_B0", 0))
    lat = np.rad2deg(np.arcsin(np.clip(yy * np.cos(b0) + z * np.sin(b0), -1, 1)))
    lon = np.rad2deg(np.arctan2(xx, z * np.cos(b0) - yy * np.sin(b0)))
    area = np.where(disk, 1 / (z * 2 * np.pi * rsun ** 2), 0)
    return disk, lat, lon, area


def dark_regions(rad, dqf, coordinates, threshold):
    disk, lat, lon, area = coordinates
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    dark = valid & (rad < threshold * reference)
    structure = np.ones((3, 3), dtype=bool)
    dark = ndimage.binary_closing(ndimage.binary_opening(dark, structure), structure) & valid
    labels, count = ndimage.label(dark, structure)
    totals = np.bincount(labels.ravel(), weights=area.ravel(), minlength=count + 1)
    keep = totals >= .0005
    keep[0] = False
    mask = keep[labels]
    labels, count = ndimage.label(mask, structure)
    return mask, labels, count, valid, lat, lon, area


def dark_area_stats(rad, dqf, header):
    # 임계 0.35/0.45/0.55 어두운 영역의 면적·위치 (180열)
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    keys = {"CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "PC1_1", "PC1_2", "PC2_1", "PC2_2"}
    h = {str(k).upper(): float(np.asarray(v).item()) for k, v in header.items() if str(k).upper() in keys}
    rsun = h["RSUN"]
    if all(key in h for key in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        scale = np.linalg.svd(np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]]), compute_uv=False).min()
        if scale <= 0:
            raise ValueError("singular SUVI PC matrix")
    else:
        scale = 1.
    radius = rsun / scale + 2       # 디스크를 덮는 정사각형으로 자름 (재표본 없음, 좌표계 유지)
    cx, cy = h["CRPIX1"] - 1, h["CRPIX2"] - 1
    x0, x1 = max(0, int(np.floor(cx - radius))), min(rad.shape[1], int(np.ceil(cx + radius)) + 1)
    y0, y1 = max(0, int(np.floor(cy - radius))), min(rad.shape[0], int(np.ceil(cy + radius)) + 1)
    rad, dqf = rad[y0:y1, x0:x1], dqf[y0:y1, x0:x1]
    h["CRPIX1"], h["CRPIX2"] = cx + 1 - x0, cy + 1 - y0
    coordinates = dark_geometry(rad.shape, h)
    out = {}
    for threshold in THRESHOLDS:
        mask, labels, count, valid, lat, lon, area = dark_regions(rad, dqf, coordinates, threshold)
        result = {"area": float(area[mask].sum()), "count": float(count),
                  "valid_fraction": float(valid.sum() / max(1, (area > 0).sum()))}
        objects = []
        for label, bounds in enumerate(ndimage.find_objects(labels), 1):
            if bounds is None:
                continue
            inside = labels[bounds] == label
            weights = area[bounds][inside]
            la, lo = lat[bounds][inside], lon[bounds][inside]
            objects.append((float(weights.sum()), float(np.average(la, weights=weights)),
                            float(np.average(lo, weights=weights)), float(np.ptp(lo)), float(np.ptp(la))))
        objects.sort(key=lambda row: (-row[0], row[1], row[2]))
        result["max_area"] = objects[0][0] if objects else 0.
        for i in range(3):
            values = objects[i] if i < len(objects) else (0.,) * 5
            result.update({f"top{i + 1}_{name}": value for name, value in
                           zip(("area", "lat", "lon", "lon_width", "lat_width"), values)})
        for width in (15, 30):
            result[f"meridian{width}_area"] = float(area[mask & (np.abs(lon) <= width)].sum())
            result[f"meridian{width}_equatorial_area"] = float(area[mask & (np.abs(lon) <= width) & (np.abs(lat) <= 30)].sum())
        result["equatorial_area"] = float(area[mask & (np.abs(lat) <= 30)].sum())
        longitude = np.clip(((lon + 90) / 15).astype(int), 0, 11)
        latitude = np.clip(((lat + 90) / 60).astype(int), 0, 2)
        totals = np.bincount((longitude * 3 + latitude)[mask], weights=area[mask], minlength=36)
        result.update({f"lon{i:02d}_lat{j}_area": float(totals[i * 3 + j]) for i in range(12) for j in range(3)})
        if not valid.any():
            result = {key: np.nan for key in result}
        out.update({f"suvi_t{round(threshold * 100):03d}_{key}": value for key, value in result.items()})
    return out


OMEGA, STRIP_HALF, LAT_HALF = 0.55, 5.0, 30.0
K_ALL = np.arange(-72, 144)
STRIP_STATS = ("logmed", "dark", "cover")


def k_name(k):
    return f"{'m' if k < 0 else 'p'}{abs(int(k)):03d}"


STRIP_COLS = [f"suvi_strip_{s}_k{k_name(k)}" for s in STRIP_STATS for k in K_ALL]
SUVI_COLS = ["suvi_disk_log_median"] + STRIP_COLS


def stripe_stats(rad, dqf, header, age_h):
    # 디스크 log 중앙값과 stripe k별 (logmed, dark, cover): 1 + 3 * len(K_ALL) 값
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    h = {str(k).upper(): v for k, v in header.items()}

    def number(key, default=None):
        value = h.get(key, default)
        if value is None:
            raise ValueError(f"missing SUVI geometry: {key}")
        return float(np.asarray(value).item())

    yy, xx = np.indices(rad.shape, dtype=float)
    xx -= number("CRPIX1") - 1
    yy -= number("CRPIX2") - 1
    theta = np.deg2rad(number("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[number("PC1_1"), number("PC1_2")], [number("PC2_1"), number("PC2_2")]])
    rsun = number("RSUN")
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    x = (matrix[0, 0] * xx + matrix[0, 1] * yy) / rsun
    y = (matrix[1, 0] * xx + matrix[1, 1] * yy) / rsun
    disk = x * x + y * y <= 0.98 ** 2
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    z = np.sqrt(np.maximum(0, 1 - x * x - y * y))
    b0 = np.deg2rad(number("SOLAR_B0", 0))
    lat = np.rad2deg(np.arcsin(np.clip(y * np.cos(b0) + z * np.sin(b0), -1, 1)))
    lon = np.rad2deg(np.arctan2(x, z * np.cos(b0) - y * np.sin(b0)))
    out = np.full(1 + 3 * len(K_ALL), np.nan)
    if not valid.any():
        out[1 + 2 * len(K_ALL):] = 0.
        return out
    reference = np.median(rad[valid])
    out[0] = np.median(np.log1p(rad[valid]))
    band = disk & (np.abs(lat) <= LAT_HALF)
    order = np.argsort(lon[band], kind="stable")
    lon_b, valid_b, rad_b = lon[band][order], valid[band][order], rad[band][order]
    centers = OMEGA * (K_ALL - age_h)
    lo = np.searchsorted(lon_b, centers - STRIP_HALF, side="left")
    hi = np.searchsorted(lon_b, centers + STRIP_HALF, side="right")
    n = len(K_ALL)
    for i in range(n):
        region = hi[i] - lo[i]
        if region <= 0:
            out[1 + 2 * n + i] = 0.
            continue
        values = rad_b[lo[i]:hi[i]][valid_b[lo[i]:hi[i]]]
        out[1 + 2 * n + i] = len(values) / region
        if len(values):
            out[1 + i] = np.median(np.log1p(values))
            out[1 + n + i] = np.mean(values < .5 * reference)
    return out


def extract_one(task):
    path, age_h = task
    try:
        rad, dqf, header = read_suvi_frame(path)
        res = {**cell_stats(rad, dqf, header), **dark_area_stats(rad, dqf, header)}
        return path, (stripe_stats(rad, dqf, header, age_h), tuple(res),
                      np.fromiter(res.values(), dtype=np.float64, count=len(res))), None
    except Exception as exc:   # 읽지 못한 파일: 그 칸은 NaN, 목록은 MANIFEST에
        return path, None, f"{type(exc).__name__}: {exc}"


files = []
for p in sorted(NC_DIR.glob("*/*/*.nc")):
    m = SUVI_NAME.search(p.name)
    if m:
        files.append((str(p), stamp(m.groups()[6:])))
nc = pd.DataFrame(files, columns=["path", "obs_end"])
nc["bin"] = bin_hour(nc["obs_end"])
nc = nc.sort_values(["bin", "obs_end"])
multi = int(nc["bin"].duplicated().sum())
nc = nc.drop_duplicates("bin", keep="last")
nc = nc[nc["bin"].isin(times)].reset_index(drop=True)
nc["age_h"] = ((pd.DatetimeIndex(nc["bin"]) - pd.DatetimeIndex(nc["obs_end"])) / HOUR).to_numpy(float)
assert ((nc["age_h"] > 0) & (nc["age_h"] <= 1)).all()
print(f"NetCDF {len(files):,}개 | 같은 칸의 중복 {multi} | 시간 축 칸에 해당 {len(nc):,}개 | {nc.obs_end.min()} ~ {nc.obs_end.max()}")

t0 = time.time()
rows, errors = {}, {}
with ProcessPoolExecutor(WORKERS, mp_context=multiprocessing.get_context("fork")) as ex:
    for i, (path, res, err) in enumerate(ex.map(extract_one, zip(nc["path"], nc["age_h"]), chunksize=16), 1):
        if err:
            errors[path] = err
        else:
            rows[path] = res
        if i % 2000 == 0 or i == len(nc):
            print(f"  {i:,}/{len(nc):,} ({time.time() - t0:.0f}s)", flush=True)
FRAME_ALL = list(next(iter(rows.values()))[1])
assert all(k == tuple(FRAME_ALL) for _, k, _ in rows.values())
FRAME_COLS = [c for c in FRAME_ALL if c != "suvi_disk_log_median"]      # 디스크 중앙값은 위 stripe 쪽 열과 같은 값
keep, disk = [FRAME_ALL.index(c) for c in FRAME_COLS], FRAME_ALL.index("suvi_disk_log_median")
assert all(np.isclose(s[0], v[disk], equal_nan=True) for s, _, v in rows.values())
empty, empty_frame = np.full(len(SUVI_COLS), np.nan), np.full(len(FRAME_COLS), np.nan)
frame = pd.DataFrame(np.stack([rows[p][0] if p in rows else empty for p in nc["path"]]), columns=SUVI_COLS, index=pd.DatetimeIndex(nc["bin"]))
frame.insert(0, "img_age_h", np.where([p in rows for p in nc["path"]], nc["age_h"].to_numpy(), np.nan))
suvi_ts = frame.reindex(times)
frame_ts = pd.DataFrame(np.stack([rows[p][2][keep] if p in rows else empty_frame for p in nc["path"]]), columns=FRAME_COLS,
                        index=pd.DatetimeIndex(nc["bin"])).reindex(times)
print(f"프레임 요약 열 {len(FRAME_COLS)} (셀 통계 {sum(c.startswith(('suvi_disk', 'suvi_lon')) for c in FRAME_COLS)}, 면적 {sum(c.startswith('suvi_t0') for c in FRAME_COLS)})")
print(f"추출 {len(rows):,}개, 실패 {len(errors)}개, {time.time() - t0:.0f}s | 프레임이 있는 칸 {suvi_ts.img_age_h.notna().mean():.1%}")
for p, e in list(errors.items())[:10]:
    print("  실패:", p, e)

NetCDF 34,900개 | 같은 칸의 중복 0 | 시간 축 칸에 해당 33,994개 | 2022-08-22 20:22:59.900000+00:00 ~ 2026-09-29 22:59:27.700000+00:00


  2,000/33,994 (448s)


  4,000/33,994 (834s)


  6,000/33,994 (1208s)


  8,000/33,994 (1574s)


  10,000/33,994 (1994s)


... (9 outputs left out)

  30,000/33,994 (5778s)


  32,000/33,994 (6151s)


  33,994/33,994 (6514s)


프레임 요약 열 327 (셀 통계 147, 면적 180)
추출 33,989개, 실패 5개, 6515s | 프레임이 있는 칸 93.1%
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2022/264/OR_SUVI-L1b-Fe195_G18_s20222641429528_e20222641429538_c20222641430144.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253400856275_e20253400856285_c20253400859414.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401056279_e20253401056289_c20253401059409.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401256282_e20253401256292_c20253401259429.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401456286_e20253401456296_c20253401459433.nc ValueError: RSUN must be positive


## 4. ENLIL 피처

`raw_to_input_df.ipynb`의 5단계와 같은 정의를 매시 칸에 적용합니다. 칸 T의 값은 `bin_hour(modelCompletionTime)`이 [T−7일, T]인 실행만 씁니다.
- dt = 도착 예상 − T(시간). 앞으로 올 것 = dt ≥ −6h, 지난 것 = −96h ≤ dt < −6h.
- `enlil_next_arr_h`, `enlil_since_arr_h`, `enlil_n_pending`, `enlil_kp`, `enlil_glancing`.

In [5]:
runs = []
for p in sorted(ENLIL_DIR.glob("WSAEnlilSimulations_*.json")):
    for s in json.loads(p.read_text()):
        if s.get("estimatedShockArrivalTime"):
            runs.append((pd.Timestamp(s["modelCompletionTime"]), pd.Timestamp(s["estimatedShockArrivalTime"]),
                         max(s.get("kp_90") or 0, s.get("kp_135") or 0, s.get("kp_180") or 0), bool(s.get("isEarthGB"))))
runs = pd.DataFrame(runs, columns=["done", "arrival", "kp", "glancing"]).drop_duplicates()
runs["bin"] = bin_hour(runs["done"])
runs = runs.sort_values("bin").reset_index(drop=True)

rb = runs["bin"].to_numpy("datetime64[ns]").astype(np.int64)
ra = pd.DatetimeIndex(runs["arrival"]).tz_convert("UTC").to_numpy("datetime64[ns]").astype(np.int64)
rk, rg = runs["kp"].to_numpy(float), runs["glancing"].to_numpy(bool)
on = times.to_numpy("datetime64[ns]").astype(np.int64)
NS_H, WEEK = 3600 * 10 ** 9, 7 * 24 * 3600 * 10 ** 9
enl = np.full((len(on), 5), np.nan)
hi = np.searchsorted(rb, on, side="right")
lo = np.searchsorted(rb, on - WEEK, side="left")
for i in range(len(on)):
    dt = (ra[lo[i]:hi[i]] - on[i]) / NS_H
    fut, past = dt >= -6, (dt < -6) & (dt >= -96)
    enl[i] = [dt[fut].min() if fut.any() else np.nan, -dt[past].max() if past.any() else np.nan, fut.sum(),
              rk[lo[i]:hi[i]][fut].max() if fut.any() else 0, float(rg[lo[i]:hi[i]][fut].all()) if fut.any() else np.nan]
ENLIL_COLS = ["enlil_next_arr_h", "enlil_since_arr_h", "enlil_n_pending", "enlil_kp", "enlil_glancing"]
enlil_ts = pd.DataFrame(enl, index=times, columns=ENLIL_COLS)
print(f"ENLIL 실행 {len(runs):,}개 ({runs.done.min()} ~ {runs.done.max()}) | 앞으로 올 실행이 있는 칸 {(enlil_ts.enlil_n_pending > 0).mean():.1%}")

ENLIL 실행 965개 (2022-07-04 20:17:00+00:00 ~ 2026-11-01 21:43:00+00:00) | 앞으로 올 실행이 있는 칸 57.2%


## 5. 조립·저장

- 한 행 = 매시 칸 `time`. 열 순서: `time`, 분할 열, `y`, ACE 22열, `img_age_h`, `suvi_disk_log_median`, stripe 648열, ENLIL 5열, 프레임 요약 327열.
- 분할 열은 `df.parquet`와 같은 정의입니다.
  - `train_origin`: 칸이 학습 origin(2022-10-01T00Z ~ F3 cutoff − 144h)인지.
  - `query_origin`, `period`: 칸이 query origin(2025-12-31T23Z ~ 2026-09-26T23Z)인지와 그 기간(F1/F2/F3, 기간 사이 origin은 null).
  - `fit_P = train_origin & (time + 144h ≤ cutoff_P)`.
- 숫자 피처는 float32, `y`는 float64입니다(zstd).

In [6]:
CUTOFF = {p: pd.Timestamp(c) for p, (_, _, c) in PERIODS.items()}
period_of = pd.Series(pd.NA, index=times, dtype="object")
for name, (first, last, _) in PERIODS.items():
    period_of[(times >= pd.Timestamp(first)) & (times <= pd.Timestamp(last)) & times.isin(query_origins)] = name

meta = pd.DataFrame({"time": times, "train_origin": times.isin(train_origins), "query_origin": times.isin(query_origins),
                     "period": pd.Categorical(period_of.to_numpy(), categories=list(PERIODS))})
for p, cut in CUTOFF.items():
    meta[f"fit_{p}"] = meta["train_origin"].to_numpy() & (times + pd.Timedelta(hours=MASK_HOURS) <= cut)
meta["y"] = y_ts.to_numpy(float)
feat = pd.concat([ace_ts, suvi_ts, enlil_ts, frame_ts], axis=1).astype(np.float32)
assert feat.index.equals(times) and feat.columns.is_unique
table = pd.concat([meta, feat.reset_index(drop=True)], axis=1)
columns = list(table.columns)

OUT.mkdir(parents=True, exist_ok=True)
path, tmp = OUT / "df_ts.parquet", OUT / "df_ts.parquet.tmp"
table.to_parquet(tmp, index=False, compression="zstd")
tmp.replace(path)
print(f"{path}: {len(table):,}행 × {len(columns)}열, {path.stat().st_size / 1e6:.1f} MB")

manifest = {
    "dataset": "input-df-ts",
    "description": "Hourly time-series model-input table built only from data-v2/raw: one row per hour bin with ACE bin values, "
                   "SUVI longitude-stripe profiles of the frame in that bin, ENLIL features, y = ACE SWEPAM speed with status 0 "
                   "(no fill) at the label time, and origin/F1/F2/F3 144 h masks.",
    "built_by": "data-v2/notebooks/raw_to_input_df_ts.ipynb",
    "built_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "smoke": SMOKE,
    "rules": {
        "time_bin": "a time t goes to hour bin floor(t, 1h) + 1h ([H-1h, H) -> H); latest record per bin; origin O may use bins <= O",
        "y": "SWEPAM bulk speed at the label time T = time (no shift), status == 0 and finite > 0, else NaN, no fill",
        "ace": "bin value (label time - 1h row); NaN when its status != 0 or it equals the product missing value; no fill, no log, no clip",
        "suvi": "only the frame in the bin (obs_end in [T-1h, T)); no walk-back, no stale or cover mask",
        "suvi_strip": "stripe k: |lon - 0.55 deg/h * (k - img_age_h)| <= 5 deg, |lat| <= 30 deg, west positive; "
                      "logmed = median log1p radiance of valid pixels, dark = fraction below 0.5 x disk median, "
                      "cover = valid pixel fraction; k = -72..143 named m072..p143",
        "enlil": "runs with bin(modelCompletionTime) in [T-7d, T]",
        "suvi_frame": "cell statistics suvi_disk_*/suvi_lonII_latJ_* and dark-area features suvi_t035/t045/t055_* of the frame in "
                      "the bin, same definitions and names as df.parquet",
        "splits": "train_origin/query_origin/period/fit_F* as in df.parquet: fit_P = train_origin & (time + 144h <= cutoff_P)",
        "processing": "no availability delay, missing-value or outlier handling",
    },
    "rows": int(len(table)), "span": [str(times[0]), str(times[-1])],
    "columns": columns,
    "ace_columns": ACE_VARS, "strip_k": [int(K_ALL[0]), int(K_ALL[-1])], "strip_stats": list(STRIP_STATS),
    "enlil_columns": ENLIL_COLS, "suvi_frame_columns": FRAME_COLS,
    "train_origins": int(meta.train_origin.sum()), "train_span": [str(train_origins[0]), str(train_origins[-1])],
    "query_origins": int(meta.query_origin.sum()), "query_span": [str(query_origins[0]), str(query_origins[-1])],
    "query_origins_by_period": {k: int(v) for k, v in meta.period.value_counts().sort_index().items()},
    "sources": {
        "ace": {"dir": str(ACE_DIR.relative_to(DATA)), **{k: v for k, v in ace_info.items()},
                "labels": [str(ace_label.index.min()), str(ace_label.index.max())]},
        "suvi": {"dir": str(NC_DIR.relative_to(DATA)), "files_used": int(len(nc)), "files_same_bin_dropped": multi,
                 "read_errors": errors, "obs_end": [str(nc.obs_end.min()), str(nc.obs_end.max())]},
        "enlil": {"dir": str(ENLIL_DIR.relative_to(DATA)), "runs": int(len(runs))},
    },
    "files": {"df_ts.parquet": {"bytes": path.stat().st_size}},
}
(OUT / "MANIFEST_ts.json").write_text(json.dumps(manifest, indent=1, default=str) + "\n")
print("MANIFEST_ts.json 저장 | train origin", manifest["train_origins"], "| query origin", manifest["query_origins"], manifest["query_origins_by_period"])

/home/t-lab01/sunrun/data-v2/input/df_ts.parquet: 36,503행 × 1012열, 135.6 MB
MANIFEST_ts.json 저장 | train origin 32665 | query origin 6457 {'F1': 2089, 'F2': 2113, 'F3': 2113}


## 6. 검증

In [7]:
ts = pd.read_parquet(path)
assert list(ts.columns) == columns and len(ts) == len(times)
ts = ts.set_index("time")

# (a) 시간 축: 1h 연속, origin 수
assert (ts.index[1:] - ts.index[:-1] == HOUR).all()
assert ts.train_origin.sum() == len(train_origins) and ts.query_origin.sum() == len(query_origins)
if not SMOKE:
    per = ts.period.value_counts().to_dict()
    assert per == {"F1": 2089, "F2": 2113, "F3": 2113}, per
print("(a) 칸", len(ts), "| train origin", int(ts.train_origin.sum()), "| query origin", int(ts.query_origin.sum()))

# (b) y = 라벨 시각의 SWEPAM status 0 속도 (이동·채움 없음), ace_speed_kms = 칸 값 (라벨 - 1h)
sw = ace_raw["swepam"]
ok0 = sw["S"].eq(0) & (sw["ace_speed_kms"] > 0)
assert np.array_equal(ts["y"].to_numpy(), sw["ace_speed_kms"].where(ok0).reindex(ts.index).to_numpy(), equal_nan=True)
lab = ace_label["ace_speed_kms"].reindex(ts.index - HOUR).to_numpy(np.float32)
assert np.array_equal(ts["ace_speed_kms"].to_numpy(), lab, equal_nan=True)
print(f"(b) y NaN {ts.y.isna().mean():.2%} | ace_speed_kms(T) = 라벨 T-1h 값")

# (c) fit 마스크: 학습 origin만, origin + 144h <= cutoff, 학습 타깃(origin + 72h)이 그 기간 첫 query origin 이전
for p, cut in CUTOFF.items():
    used = ts.index[ts[f"fit_{p}"]]
    assert ts.loc[used, "train_origin"].all() and (used + pd.Timedelta(hours=MASK_HOURS) <= cut).all()
    assert (used + pd.Timedelta(hours=H) < pd.Timestamp(PERIODS[p][0])).all()
    print(f"(c) {p}: 학습 origin {len(used):,}개 (마지막 {used.max() if len(used) else None}), cutoff {cut}")

# (d) SUVI: img_age_h 범위, cover 범위, 프레임이 없는 칸은 전부 NaN
age = ts["img_age_h"].dropna()
assert ((age > 0) & (age <= 1)).all()
strip = ts[STRIP_COLS]
cover = ts[[c for c in STRIP_COLS if "_cover_" in c]]
assert strip[ts.img_age_h.isna()].isna().all().all()
assert ((cover.dropna(how="all") >= 0) & (cover.dropna(how="all") <= 1)).all().all()
print(f"(d) 프레임 있는 칸 {ts.img_age_h.notna().mean():.1%} | stripe NaN(프레임 있는 칸) {strip[ts.img_age_h.notna()].isna().mean().mean():.2%}")

# (e) ENLIL 인과성: 표본 칸에서 n_pending을 raw 실행 목록으로 다시 계산
rng = np.random.default_rng(0)
sample = pd.DatetimeIndex(rng.choice(ts.index, size=min(300, len(ts)), replace=False))
n_pend = [int(((runs["bin"] <= o) & (runs["bin"] >= o - pd.Timedelta(days=7)) & ((runs["arrival"] - o) / HOUR >= -6)).sum())
          for o in sample]
assert np.array_equal(ts.loc[sample, "enlil_n_pending"].to_numpy(), np.asarray(n_pend, dtype=np.float32))
print(f"(e) 표본 {len(sample)}칸: enlil_n_pending이 raw 재계산과 일치")

# (f) df.parquet와 교차 확인 (확인용으로만 읽음): 같은 origin의 ACE _last, img_age_h, ENLIL, fit 마스크, (origin, h)의 y
DF = DATA / "input/df.parquet"
if DF.exists() and not SMOKE:
    last_cols = [f"{c}_last" for c in ACE_VARS]
    ref = pd.read_parquet(DF, columns=["set", "origin", "fit_F1", "fit_F2", "fit_F3", "img_age_h", "suvi_disk_log_median",
                                       "suvi_lon06_lat1_log_median", "suvi_lon07_lat1_log_median", *ENLIL_COLS, *last_cols],
                          filters=[("h", "=", 1)])
    one = ref.drop_duplicates("origin").set_index("origin").sort_index()
    mine = ts.loc[one.index]
    for c in ACE_VARS:
        assert np.array_equal(mine[c].to_numpy(), one[f"{c}_last"].to_numpy(), equal_nan=True), c
    for c in ENLIL_COLS:
        assert np.allclose(mine[c].to_numpy(), one[c].to_numpy(), equal_nan=True, rtol=1e-6, atol=1e-6), c
    # SUVI 프레임: 둘 다 있는 칸은 같은 값. df_ts에만 있는 칸은 df.parquet를 만든 뒤 raw에 추가된 프레임이고,
    # df.parquet에만 있는 칸은 읽지 못한 프레임이다(df.parquet는 그 칸에도 img_age_h를 남기고, df_ts는 NaN으로 둔다).
    both = mine["img_age_h"].notna() & one["img_age_h"].notna()
    only_ts = mine["img_age_h"].notna() & one["img_age_h"].isna()
    only_df = mine["img_age_h"].isna() & one["img_age_h"].notna()
    assert np.allclose(mine.loc[both, "img_age_h"], one.loc[both, "img_age_h"], rtol=1e-6, atol=1e-6)
    assert np.allclose(mine.loc[both, "suvi_disk_log_median"], one.loc[both, "suvi_disk_log_median"], equal_nan=True, rtol=1e-5, atol=1e-5)
    error_bins = set(nc.loc[nc["path"].isin(list(errors)), "bin"])
    assert set(one.index[only_df]) <= error_bins, sorted(set(one.index[only_df]) - error_bins)[:5]
    print(f"    SUVI 프레임: 둘 다 {int(both.sum()):,}칸 일치 | df_ts에만 {int(only_ts.sum()):,}칸"
          f" ({one.index[only_ts].min()} ~ {one.index[only_ts].max()})" if only_ts.any() else "    SUVI 프레임: 전부 일치",
          f"| 읽지 못해 df_ts에서 NaN인 칸 {int(only_df.sum())}")
    tr = ref[ref["set"] == "train"].set_index("origin")
    for p in PERIODS:
        assert np.array_equal(ts.loc[tr.index, f"fit_{p}"].to_numpy(), tr[f"fit_{p}"].to_numpy()), p
    assert set(tr.index) == set(ts.index[ts.train_origin]) and set(ref[ref["set"] == "query"].origin) == set(ts.index[ts.query_origin])
    yy = pd.read_parquet(DF, columns=["target_time", "y"])
    assert np.array_equal(ts["y"].reindex(pd.DatetimeIndex(yy["target_time"])).to_numpy(), yy["y"].to_numpy(), equal_nan=True)
    # stripe와 15도 셀: 셀 lon06(0~15도)·lon07(15~30도) 중심의 stripe logmed가 셀 중앙값과 같이 움직이는지
    # 프레임 요약 열(셀 통계·면적): 둘 다 프레임이 있는 칸에서 df.parquet와 같은 값
    ref_frame = (pd.read_parquet(DF, columns=["origin", *FRAME_COLS], filters=[("h", "=", 1)]).drop_duplicates("origin")
                 .set_index("origin").sort_index())
    assert ref_frame.index.equals(one.index)
    same = mine.loc[both.to_numpy(), FRAME_COLS].to_numpy()
    assert np.allclose(same, ref_frame.loc[both.to_numpy(), FRAME_COLS].to_numpy(), equal_nan=True, rtol=1e-5, atol=1e-6)
    print(f"    프레임 요약 {len(FRAME_COLS)}열: 둘 다 프레임이 있는 {int(both.sum()):,}칸에서 df.parquet와 일치")
    for cell_name, center in (("suvi_lon06_lat1_log_median", 7.5), ("suvi_lon07_lat1_log_median", 22.5)):
        k = int(round(center / OMEGA))
        pair = pd.concat([mine[f"suvi_strip_logmed_k{k_name(k)}"].rename("stripe"), one[cell_name].rename("cell")], axis=1).dropna()
        corr = float(np.corrcoef(pair.iloc[:, 0], pair.iloc[:, 1])[0, 1])
        print(f"    stripe k={k} (경도 {OMEGA * k:.1f}도) vs {cell_name}: 상관 {corr:.3f}")
        assert corr > .9, (cell_name, corr)
    print(f"(f) df.parquet 교차 확인: origin {len(one):,}개의 ACE·img_age_h·disk·ENLIL·fit·y 일치")
else:
    print("(f) df.parquet 교차 확인 생략 (SMOKE 또는 파일 없음)")

# (g) 이름: ch 토큰 없음, 열 중복 없음, 결측 비율
assert not [c for c in columns if re.search(r"(^|_)ch(_|$)", c.lower())]
assert len(set(columns)) == len(columns)
nan_rate = ts.drop(columns=["train_origin", "query_origin", "period", "fit_F1", "fit_F2", "fit_F3"]).isna().mean()
display(nan_rate.groupby(lambda c: "y" if c == "y" else "ace" if c.startswith("ace_") else "enlil" if c.startswith("enlil_") else "suvi")
        .describe().round(3))

(a) 칸 36503 | train origin 32665 | query origin 6457
(b) y NaN 32.61% | ace_speed_kms(T) = 라벨 T-1h 값
(c) F1: 학습 origin 28,297개 (마지막 2025-12-23 00:00:00+00:00), cutoff 2025-12-29 00:00:00+00:00
(c) F2: 학습 origin 30,457개 (마지막 2026-03-23 00:00:00+00:00), cutoff 2026-03-29 00:00:00+00:00
(c) F3: 학습 origin 32,665개 (마지막 2026-06-23 00:00:00+00:00), cutoff 2026-06-29 00:00:00+00:00
(d) 프레임 있는 칸 93.1% | stripe NaN(프레임 있는 칸) 0.01%


(e) 표본 300칸: enlil_n_pending이 raw 재계산과 일치


    SUVI 프레임: 둘 다 32,983칸 일치 | df_ts에만 378칸 (2022-10-01 00:00:00+00:00 ~ 2022-10-18 20:00:00+00:00) | 읽지 못해 df_ts에서 NaN인 칸 4


    프레임 요약 327열: 둘 다 프레임이 있는 32,983칸에서 df.parquet와 일치
    stripe k=14 (경도 7.7도) vs suvi_lon06_lat1_log_median: 상관 0.992
    stripe k=41 (경도 22.6도) vs suvi_lon07_lat1_log_median: 상관 0.992
(f) df.parquet 교차 확인: origin 34,968개의 ACE·img_age_h·disk·ENLIL·fit·y 일치


,count,mean,std,min,25%,50%,75%,max
ace,22.0,0.200,0.199,0.000,0.146,0.175,0.178,1.000
enlil,5.0,0.241,0.223,0.000,0.000,0.351,0.428,0.428
suvi,977.0,0.073,0.039,0.069,0.069,0.069,0.069,0.431
y,1.0,0.326,NaN,0.326,0.326,0.326,0.326,0.326


## 7. 사용법

```python
import pandas as pd
ts = pd.read_parquet("/home/t-lab01/sunrun/data-v2/input/df_ts.parquet").set_index("time")
O = pd.Timestamp("2026-01-10T00:00Z")
window = ts.loc[O - pd.Timedelta(hours=119):O]                    # origin O의 5일 lookback (칸 <= O)
target = ts.loc[O + pd.Timedelta(hours=1):O + pd.Timedelta(hours=72), "y"]   # 리드 1..72의 정답
sweep = ts.loc[O, [f"suvi_strip_logmed_kp{k:03d}" for k in range(120)]]      # 칸 O 프레임 한 장의 경도 스윕
```

- 이 표에는 결측 처리가 없습니다. 창 안 앞값 채움, 결측 마스크, 표준화, 프레임 되돌아가기는 `research/_module/seq.py`가 합니다.
- stripe k와 리드의 관계: 속도 v의 태양풍이 O + h에 L1에 닿으려면 k = τ_v − h (τ_v = (1 AU − 2.5 R☉)/v, 350 km/s에서 약 117h, 750 km/s에서 약 55h)인 stripe를 떠났어야 합니다.